In [18]:
import pickle
import pandas as pd
import numpy as np
import csv

In [19]:
lib_path = '/home/liuycomputing/wsp_sequencing/FINAL.xlsx'
file_folder = '/home/liuycomputing/wsp_sequencing/seq_20250115/'
error_file_name = 'whole_error_dict_read2_20250117.pkl'
per_seq_filename = 'per_item_num_read2.csv'
per_item_filename = 'per_item_error_rate_read2.csv'

In [20]:
def load_error_dict_pickle(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f)
    
def read_library(file_path):  
    return pd.read_excel(file_path, header=None).iloc[:, 0]

# 读取文库列表，从索引1开始
library = read_library(lib_path)
whole_read_error_dict = load_error_dict_pickle(file_path=file_folder+error_file_name)
print('总序列数：', len(library))

总序列数： 34680


In [21]:
zero_num = 0
no_index_list = []
for i, ideal_seq in enumerate(library):
    right_list = np.array(whole_read_error_dict[i]['Right'], dtype=float)
    deletion_list = np.array(whole_read_error_dict[i]['Deletion'], dtype=float)
    insertion_list = np.array(whole_read_error_dict[i]['Insertion'], dtype=float)
    substitution_list = np.array(whole_read_error_dict[i]['Substitution'], dtype=float)
    total_num = right_list + deletion_list + insertion_list + substitution_list

    if total_num[0] < 1:
        no_index_list.append(i)
        zero_num += 1

print('缺失序列数：', zero_num)
print('缺失序列：', no_index_list)

缺失序列数： 758
缺失序列： [5, 20, 37, 45, 82, 102, 120, 567, 578, 581, 582, 756, 839, 840, 1242, 1244, 1349, 1481, 1499, 1563, 1610, 1666, 1708, 1904, 2041, 2062, 2071, 2207, 2377, 2413, 2414, 2479, 2602, 2603, 2646, 2725, 2799, 2823, 2867, 2869, 2890, 2920, 2939, 2952, 2970, 3028, 3118, 3241, 3378, 3401, 3431, 3450, 3533, 3541, 3548, 3708, 3742, 3793, 3828, 3843, 3856, 3898, 3952, 3962, 3990, 4063, 4066, 4142, 4159, 4196, 4219, 4220, 4271, 4316, 4400, 4414, 4452, 4474, 4497, 4539, 4574, 4644, 4653, 4674, 4680, 4701, 4755, 4761, 4864, 4870, 4913, 4933, 4953, 4959, 5010, 5023, 5108, 5187, 5191, 5233, 5269, 5326, 5527, 5568, 5648, 5694, 5848, 5901, 5906, 5939, 5953, 6007, 6022, 6029, 6189, 6287, 6400, 6447, 6450, 6456, 6512, 6538, 6648, 6720, 6728, 6748, 6758, 6825, 6841, 6870, 6892, 6913, 6979, 6999, 7005, 7022, 7024, 7102, 7108, 7159, 7188, 7211, 7256, 7284, 7298, 7299, 7306, 7371, 7375, 7465, 7476, 7482, 7511, 7524, 7542, 7557, 7580, 7605, 7734, 7746, 7780, 7788, 7797, 7868, 7980, 8030, 8040, 

In [22]:
def write_file(file_name, error_dict, number):
    with open(file_name, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)

        # 写入表头
        header = ['Key', 'Error Type'] + [f'Position {i+1}' for i in range(number)]
        writer.writerow(header)

        # 写入数据
        for key, error_types in error_dict.items():
            for error_type, values in error_types.items():
                row = [key, error_type] + values
                writer.writerow(row)
    
    print(file_name, ' done')

lib_len = len(library[0])

def compute_error_rate(library, error_dict):
    # 初始化整个集的lib
    lib_len = len(library[0])
    complete_set_dict = {}
    complete_set_dict['Right'] = np.zeros(lib_len)
    complete_set_dict['Deletion'] = np.zeros(lib_len)
    complete_set_dict['Insertion'] = np.zeros(lib_len)
    complete_set_dict['Substitution'] = np.zeros(lib_len)
    complete_set_dict['Total'] = np.zeros(lib_len)

    # 计算错误率
    rate_dict, global_error_dict = {}, {}
    for i, ideal_seq in enumerate(library):
        rate_dict[i] = {}
        global_error_dict[i] = {}

        right_list = np.array(error_dict[i]['Right'], dtype=float)
        deletion_list = np.array(error_dict[i]['Deletion'], dtype=float)
        insertion_list = np.array(error_dict[i]['Insertion'], dtype=float)
        substitution_list = np.array(error_dict[i]['Substitution'], dtype=float)
        total_num = right_list + deletion_list + insertion_list + substitution_list

        if np.mean(total_num) != 0:
            rate_dict[i]['Right-Rate'] = (right_list / total_num).tolist()
            rate_dict[i]['Deletion-Rate'] = (deletion_list / total_num).tolist()
            rate_dict[i]['Insertion-Rate'] = (insertion_list / total_num).tolist()
            rate_dict[i]['Substitution-Rate'] = (substitution_list / total_num).tolist()

            global_error_dict[i]['Right'] = (right_list).tolist()
            global_error_dict[i]['Deletion'] = (deletion_list).tolist()
            global_error_dict[i]['Insertion'] = (insertion_list).tolist()
            global_error_dict[i]['Substitution'] = (substitution_list).tolist()
            global_error_dict[i]['Total'] = (total_num).tolist()

        else:
            rate_dict[i]['Right-Rate'] = [0] * lib_len
            rate_dict[i]['Deletion-Rate'] = [0] * lib_len
            rate_dict[i]['Insertion-Rate'] = [0] * lib_len
            rate_dict[i]['Substitution-Rate'] = [0] * lib_len

            global_error_dict[i]['Right'] = [0] * lib_len
            global_error_dict[i]['Deletion'] = [0] * lib_len
            global_error_dict[i]['Insertion'] = [0] * lib_len
            global_error_dict[i]['Substitution'] = [0] * lib_len
            global_error_dict[i]['Total'] = [0] * lib_len

        complete_set_dict['Right'] += right_list
        complete_set_dict['Deletion'] += deletion_list
        complete_set_dict['Insertion'] += insertion_list
        complete_set_dict['Substitution'] += substitution_list
        complete_set_dict['Total'] += total_num
        
    return rate_dict, complete_set_dict, global_error_dict

# 分条错误率
rate_dict, complete_set_dict, global_error_dict = compute_error_rate(library, whole_read_error_dict)

# 保存分条计数文件
write_file(file_name= file_folder+per_seq_filename, 
            error_dict=global_error_dict, number=lib_len)

# 保存分条错误率文件
write_file(file_name= file_folder+per_item_filename, 
            error_dict=rate_dict, number=lib_len)

/home/liuycomputing/wsp_sequencing/seq_20250115/per_item_num_read2.csv  done
/home/liuycomputing/wsp_sequencing/seq_20250115/per_item_error_rate_read2.csv  done
